# FiGuard + CrewAI

**`FiGuardCrewGuard`** wraps individual CrewAI tools with FiGuard pre-flight authorization. Each tool call is checked against a budget before it runs — if the budget is exhausted or the category is wrong, the tool returns a structured denial string instead of executing.

This notebook covers:
1. **Single agent** — wrap one tool, enforce a shared budget
2. **Fleet** — each sub-agent gets its own delegation token with a hard cap; a rogue agent can only exhaust its own allocation

Run against the live sandbox — no account required.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/figuard/figuard-notebooks/blob/main/crewai-research-fleet.ipynb)

In [ ]:
import sys
!pip install "figuard[crewai]>=0.3.2" --upgrade -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} + CrewAI ready")

## Part 1 — Single Agent

`FiGuardCrewGuard` wraps a tool **in-place**. Pass the same tool object to your `Agent` — the guard patches its `_run` method so every call goes through FiGuard first.

In [ ]:
from crewai.tools import BaseTool
from figuard.integrations.crewai import FiGuardCrewGuard
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

budget = client.create_budget(
    user_id="travel_agent",
    total_limit=500.00,
    currency="USD",
    expires_in="1h",
)

# Extract session token from tokens array (one entry per dimension)
tokens = {t.category: t.session_token for t in budget.tokens}
session_token = tokens["default"]  # simple budget uses "default" category

print(f"Budget: ${budget.total_limit:.2f}  id={budget.id}")
print()

# --- Define a minimal tool for the demo ---
class BookFlightTool(BaseTool):
    name: str = "book_flight"
    description: str = "Book a flight given a destination and price."

    def _run(self, destination: str, amount: float) -> str:
        return f"Booked flight to {destination} for ${amount:.2f}"

book_flight_tool = BookFlightTool()

# Wrap the tool — modifies _run in-place
FiGuardCrewGuard(
    tool=book_flight_tool,
    client=client,
    session_token=session_token,
    agent_id="travel_agent",
    category="flight",
    amount_key="amount",
)

# Authorized call
result1 = book_flight_tool._run(destination="SFO→JFK", amount=267.00)
print(f"Call 1 (within budget): {result1}")

# Denied call — would exceed budget
result2 = book_flight_tool._run(destination="SFO→LHR", amount=450.00)
print(f"Call 2 (over budget):   {result2}")

print(f"\n→ Ledger: https://figuard-sandbox-1.onrender.com/ui")

## Part 2 — Fleet with Delegation Tokens

In a multi-agent fleet each sub-agent gets a **delegation token** with its own cap. A rogue agent that loops or overspends can only exhaust its own allocation — the rest of the fleet is unaffected.

The wrapping pattern is the same as single-agent: one `FiGuardCrewGuard` call per tool, using that agent's delegation token.

In [ ]:
from crewai.tools import BaseTool
from figuard.integrations.crewai import FiGuardCrewGuard
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

# Fleet budget — the total ceiling across all agents
fleet = client.create_budget(
    user_id="orchestrator",
    total_limit=1000.00,
    currency="USD",
    expires_in="2h",
)

# Extract session token from tokens array
fleet_tokens = {t.category: t.session_token for t in fleet.tokens}
fleet_session_token = fleet_tokens["default"]


# Each sub-agent gets a scoped delegation token
researcher_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet_session_token,
    label="researcher",
    caps=[{"category": "search", "limit": 200.00}],
    expires_in="2h",
)
writer_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet_session_token,
    label="writer",
    caps=[{"category": "llm", "limit": 300.00}],
    expires_in="2h",
)
print(f"Fleet: ${fleet.total_limit:.2f}  |  researcher $200  writer $300")
print()

# --- Define tools ---
class SearchTool(BaseTool):
    name: str = "search"
    description: str = "Search the web for information."
    def _run(self, query: str, amount: float = 5.0) -> str:
        return f"Search results for: {query}"

class WriteTool(BaseTool):
    name: str = "write_report"
    description: str = "Write a research report."
    def _run(self, topic: str, amount: float = 50.0) -> str:
        return f"Report on: {topic}"

researcher_tool = SearchTool()
writer_tool     = WriteTool()

# Wrap each tool with its agent's delegation token
FiGuardCrewGuard(
    tool=researcher_tool,
    client=client,
    session_token=researcher_token.session_token,
    agent_id="researcher",
    category="search",
)
FiGuardCrewGuard(
    tool=writer_tool,
    client=client,
    session_token=writer_token.session_token,
    agent_id="writer",
    category="llm",
)

# Researcher burns its allocation
print("Researcher (tight loop — simulating rogue behavior)...")
for i in range(1, 50):
    result = researcher_tool._run(query=f"query {i}", amount=5.0)
    if result.startswith("FiGuard DENIED"):
        print(f"  call {i:2d}: {result}")
        print(f"  ✓ Researcher capped at $200 — stopped at call {i}")
        break
    if i <= 2 or i % 10 == 0:
        print(f"  call {i:2d}: AUTHORIZED")

print()

# Writer is unaffected
result = writer_tool._run(topic="AI safety trends", amount=50.0)
print(f"Writer: {result}")
print("✓ Writer unaffected — its $300 allocation is independent")

print(f"\n→ Ledger: https://figuard-sandbox-1.onrender.com/ui")